<img src="images/m-rainbow.svg" width="5%" height="5%">

<h1 style="font-size: 30px; font-weight: bold; color: #ff2f05;">
  The Mistral AI Python SDK
</h1>

The Mistral AI Python SDK (**S**oftware **D**evelopment **K**it) is a wrapper for the **Mistral AI API**.

You can find the official documentation and some examples in:
- The [Vibe Studio Product Section](https://docs.mistral.ai/studio-api/overview) 
- The [API reference](https://docs.mistral.ai/api)
- The [Developers Section](https://docs.mistral.ai/developers)
- Their Github [Python SDK](https://github.com/mistralai/client-python) and [Cookbook](https://github.com/mistralai/cookbook) repositories
- Their [YouTube Streams](https://www.youtube.com/@MistralAIOfficial/streams)

<h2 style="font-size: 25px; font-weight: bold; color: #fb6227;">
  7. Conversations with Tools
</h2>

The main complication when working with built-in tools is to process entries (from the `ConversationResponse.outputs` list) and display them correctly to the end-users.

The documentation includes details on how the response is formatted for each tool and here's an overview:

| Capability | Initialization (class or dict) | Expected Output|
| --- | --- | --- |
| [Web Search](https://docs.mistral.ai/studio-api/agents/agent-tools/websearch)| WebSearchTool()<br><br>{"type": "web_search"} | - **ToolExecutionEntry**: Execution of the web search<br>- **MessageOutputEntry**: Generated answer from the model (content can be **TextChunk** or **ToolReferenceChunk**) |
| [Document Libraries](https://docs.mistral.ai/studio-api/knowledge-rag/libraries#connecting-libraries-to-agents)| DocumentLibraryTool(library_ids=["..."])<br><br>{"type": "document_library", "library_ids": ["..."]} | - **ToolExecutionEntry**: Execution of the library search<br>- **MessageOutputEntry**: Generated answer from the model (content can be **TextChunk** or **ToolReferenceChunk**) |
| [Code Interpreter](https://docs.mistral.ai/studio-api/agents/agent-tools/code_interpreter) | CodeInterpreterTool()<br><br>{"type": "code_interpreter"} | - **MessageOutputEntry**: Initial response from the assistant, indicating that it can help generate the code<br>- **ToolExecutionEntry**: Execution of the code interpreter tool. Look at `info` to find the code and code output<br>- **MessageOutputEntry**: Generated answer from the model (content can be **ToolFileChunk** with a `file_id` to download the file or **TextChunk**) |
| [Image Generation](https://docs.mistral.ai/studio-api/agents/agent-tools/image_generation) | ImageGenerationTool()<br><br>{"type": "image_generation"} | - **ToolExecutionEntry**: Execution of the image generation tool<br>- **MessageOutputEntry**: Generated answer from the model (content can be **TextChunk** or **ToolReferenceChunk**) |
| [Functions Calling](https://docs.mistral.ai/studio-api/agents/agent-tools/function-calling) | Define functions as per the [OpenAI format](https://developers.openai.com/api/docs/guides/function-calling#defining-functions)| List of **FunctionCallEntry**; you need to generate a **FunctionResultEntry** for each and append them to the conversation|

In [1]:
from mistralai.client import Mistral
from mistralai.client.models import UserMessage
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.environ["MISTRAL_API_KEY"]
mistral = Mistral(api_key=api_key)

In [2]:
from mistralai.client.models import MessageOutputEntry, ToolExecutionEntry, TextChunk, ToolReferenceChunk, ToolFileChunk
from IPython.display import display, Markdown, Image
import json

# This function is continuous work-in-progress and may evolve when coming across new tools/scenarios
def display_entries(entries: list, download_files: bool = False) -> None:
    """
    Function to display entries in a controlled manner

    args:
      entries: list
          Conversation entries from the ConversationResponse.outputs list
      download_files: bool (default = False)
          Whether to download in the cwd files found in the MessageOutputEntry objects
          May be useful as the LLM sometimes assumes that these files are available
          and tries to display them.
    """

    # These lists are used to collect various outputs before we decide in which order to display them 
    assistant_list = []
    additional_content = []

    # We use a set for references to avoid duplicates and because the order is not important
    references = set()

    # Loop through all entries 
    for entry in entries:

        # OUTPUT MESSAGES ----------------------------------------------------------------------------------------------
        if isinstance(entry, MessageOutputEntry):

            # If the message content is a single string, we just display it
            if isinstance(entry.content, str) and entry.content!=".":
                assistant_list.append(Markdown(entry.content))
                continue

            # If the message content consists of multiple content chunks, we need to loop through them
            for chunk in entry.content:
                if isinstance(chunk, TextChunk) and chunk.text!=".":
                    assistant_list.append(Markdown(chunk.text))
                elif isinstance(chunk, ToolReferenceChunk):
                    match chunk.tool:
                        case "web_search":
                            references.add(f"[{chunk.title}]({chunk.url})<br>")
                        case "document_library":
                            references.add(f"[{chunk.title}](https://chat.mistral.ai/libraries/019e93b2-8cd4-76a4-bded-ebe721d359a5?document={chunk.url})<br>")
                        case "code_interpreter":
                            pass
                elif isinstance(chunk, ToolFileChunk) and download_files==True:
                    # Some images had a filename without extension
                    extension = f".{chunk.file_type}"
                    if not chunk.file_name.endswith(extension):
                        filename = f"{chunk.file_name}{extension}"
                    else:
                        filename = chunk.file_name
                    # Download file in current working directory (useful when the LLM tries to display files but only provides the filename)
                    file_content = mistral.files.download(file_id=chunk.file_id).read() 
                    with open(f"{filename}", "wb") as f:
                        f.write(file_content)
                    
        
        # TOOL EXECUTION MESSAGES --------------------------------------------------------------------------------------
        elif isinstance(entry, ToolExecutionEntry):

            match entry.name:
                case "web_search":
                    query = json.loads(entry.arguments)["query"]
                    display(Markdown(f"*⏳ Searching the web for `{query}`...*"))
                case "code_interpreter":
                    additional_content.append(Markdown(f"```text {entry.info["code"]}"))
                    results = entry.info["result"]
                    for result in results:
                        if result["type"]=="file_url":
                            additional_content.append(Image(url=result["file_url"]))
                case "image_generation":
                    additional_content.append(Image(url=json.loads(entry.info["result"])["url"]))
    
    # Display the assistant messages on top
    display(Markdown(f"**🤖 ASSISTANT** {'-' * 100}"))
    for content in assistant_list:
        display(content)

    # Continue with additional content to show in the conversation (images, code samples, charts)
    for content in additional_content:
        display(content)
    
    # Finish with references/citations if any.
    # Only strings are stored in the set so we can avoid duplicates (saving Markdown() objects doesn't work as each object is different even if their representation is the same)
    if references:
        display(Markdown(f"**📚 REFERENCES** {'-' * 100}"))
        for content in references:
            display(Markdown(content))

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  7.1 Web Search (built-in)
</h3>

In [3]:
from mistralai.client.models import WebSearchTool

search_response = mistral.beta.conversations.start(
    model="mistral-small-latest",
    instructions="You are a journalist looking for fresh news with your web_search tool",
    tools=[WebSearchTool()], # or {"type": "web_search"}
    inputs="What is the biggest financial news that happened on June 12th 2026?"
)

search_response.outputs

[ToolExecutionEntry(name='web_search', arguments='{"query": "biggest financial news June 12 2026"}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 6, 16, 16, 0, 47, 746550, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 6, 16, 16, 0, 48, 978960, tzinfo=TzInfo(0)), agent_id=Unset(), model='mistral-small-latest', id='tool_exec_019ed129f28275ce82c5431f98878565', info={'result': '{"0": {"url": "https://www.caixabankresearch.com/en/publications/financial-markets-daily-report/12-june-2026", "title": "Financial Markets Daily Report 12 June 2026", "description": "Yesterday&#x27;s session was marked ... with Tehran. <strong>Brent crude prices dropped nearly 3%, to settle just above $90/bbl, while gold rebounded more than 3%, being priced above $4200/ounce at session-ending.</strong>...", "snippets": ["Yesterday&#x27;s session was marked ... with Tehran. <strong>Brent crude prices dropped nearly 3%, to settle just above $90/bbl, while gold rebounded more than

In [4]:
display_entries(search_response.outputs)

*⏳ Searching the web for `biggest financial news June 12 2026`...*

**🤖 ASSISTANT** ----------------------------------------------------------------------------------------------------

The biggest financial news on June 12, 2026, was the **SpaceX initial public offering (IPO)**, which became the largest IPO in history at $75 billion. The company began trading on the Nasdaq, and shadow markets anticipated a surge of at least 35% on its debut. This event led to strong gains in US equities, with the S&P 500 up 1.8% and the Nasdaq up 2.5%

.

Additionally, oil prices fell sharply after reports of a potential US-Iran deal, while gold rebounded over 3% to above $4,200 per ounce

**📚 REFERENCES** ----------------------------------------------------------------------------------------------------

[The Week That Was: June 12, 2026](https://www.cnbc.com/video/2026/06/12/the-week-that-was-june-12-2026.html)<br>

[Investment Update on Financial Market News – 12 June 2026 — Patronus Partners](https://www.patronuspartners.com/daily-insights/jls3z7jejwezwra5dpttey673cc4pp)<br>

[Financial Markets Daily Report 12 June 2026](https://www.caixabankresearch.com/en/publications/financial-markets-daily-report/12-june-2026)<br>

[The Pre-Market Rundown 2: June, 12, 2026](https://www.cnbc.com/video/2026/06/12/the-pre-market-rundown-2-june-12-2026.html)<br>

[Markets News, June 2, 2026: AI Trade Keeps Rocking as Marvell, HPE Soar; Major U.S. Indexes Close at Fresh Record Highs](https://www.investopedia.com/stock-market-today-dow-jones-s-and-p-500-06022026-11988714)<br>

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  7.2 Document Libraries (built-in)
</h3>

In Mistral Vibe, you can find your library ID in the URL

<img src="images/library_url.png" width="50%" height="50%">

We'll cover in another video how to work with libraries in details (e.g. create or retrieve a library programatically within the Python SDK)

In [5]:
from mistralai.client.models import DocumentLibraryTool

lib = DocumentLibraryTool(library_ids=["019e93b2-8cd4-76a4-bded-ebe721d359a5"]) # or {"type": "document_library", "library_ids": ["019e93b2-8cd4-76a4-bded-ebe721d359a5"]}

lib_response = mistral.beta.conversations.start(
    model="mistral-small-latest",
    inputs="Can you find the document about IFRS 16?", 
    tools=[lib]
)

lib_response.outputs

[ToolExecutionEntry(name='document_library', arguments='{"query": "IFRS 16", "libraries_ids": ["019e93b2-8cd4-76a4-bded-ebe721d359a5"]}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 6, 16, 16, 3, 23, 467984, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 6, 16, 16, 3, 23, 942059, tzinfo=TzInfo(0)), agent_id=Unset(), model='mistral-small-latest', id='tool_exec_019ed12c52cb71d2b255bff7a8f3836c', info={'result': '{"0": {"url": "13b24aca-9fba-4d2d-acb2-7cc245a7fbb7", "title": "ifrs-16-leases.pdf", "description": "IFRS 16, effective since January 2019, replaces IAS 17 and sets principles for lease recognition, measurement, presentation, and disclosure for both lessees and lessors. It mandates lessees to recognize most leases on the balance sheet as right-of-use assets and lease liabilities, with exemptions for short-term and low-value leases. The standard includes guidance on lease modifications, sale-and-leaseback transactions, and recent amendments a

In [6]:
display_entries(lib_response.outputs)

**🤖 ASSISTANT** ----------------------------------------------------------------------------------------------------

I found a document about **IFRS 16 (Leases)**. Here are the key details:

### **Title**: IFRS 16 - Leases
### **Description**:
IFRS 16, effective since January 2019, replaces **IAS 17** and establishes principles for lease recognition, measurement, presentation, and disclosure for both **lessee**s and **lessor**s. It requires lessees to recognize most leases on the balance sheet as **right-of-use assets** and **lease liabilities**, with exemptions for **short-term** and **low-value leases**. The standard includes guidance on:
- **Lease modifications**
- **Sale-and-leaseback transactions**
- **COVID-19 rent concessions**
- **Interest rate benchmark reforms**

### **Key Sections**:
1. **Objective & Scope** (Paragraphs 1–5)
   - Applies to all leases except for:
     - Mineral/oil/gas leases
     - Biological assets (IAS 41)
     - Service concession arrangements (IFRIC 12)
     - Licensing agreements (e.g., intellectual property)
   - Lessees may elect **not to apply IFRS 16** to short-term or low-value leases

.

2. **Lessee Accounting** (Paragraphs 22–49)
   - **Recognition**: Lessees must recognize a **right-of-use asset** and a **lease liability** (except for exemptions)

.
   - **Measurement**:
     - **Lease liability**: Discounted present value of lease payments (using the **implicit rate** or **incremental borrowing rate**).
     - **Right-of-use asset**: Initially measured at cost, then subject to depreciation/impairment

.
   - **Exemptions**: Short-term leases (<12 months) and low-value assets (e.g., laptops, small office equipment)

...

.

6. **Disclosures** (Paragraphs 51–60)
   - Lessees must disclose:
     - **Maturity analysis** of lease liabilities.
     - **Nature of leasing activities** (e.g., variable payments, extension/termination options).
     - **Restrictions/covenants** imposed by leases

.

7. **Recent Amendments**:
   - **Interest Rate Benchmark Reform (Phase 2)** – Adjustments for hedging relationships.
   - **Lease Liability in Sale and Leaseback** – Clarifies accounting for transactions after the transition date

.

---
### **Document Reference**:
📄 **[IFRS 16 - Leases (Full Text)](13b24aca-9fba-4d2d-acb2-7cc245a7fbb7)**
*(Includes illustrative examples, transition guidance, and amendments.)*

Would you like a summary of a specific section (e.g., **lessee vs. lessor accounting**, **sale-and-leaseback**, or **disclosures**)?

**📚 REFERENCES** ----------------------------------------------------------------------------------------------------

[ifrs-16-leases.pdf](https://chat.mistral.ai/libraries/019e93b2-8cd4-76a4-bded-ebe721d359a5?document=13b24aca-9fba-4d2d-acb2-7cc245a7fbb7)<br>

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  7.3 Code Interpreter (built-in)
</h3>

In [7]:
# Check the capabilities without tools
test_response = mistral.beta.conversations.start(
    model="mistral-small-latest",
    inputs="Tell me what is the current time in Paris",
)

test_response.outputs

[MessageOutputEntry(content="I don't have real-time capabilities to provide the current time in Paris. I recommend checking a world clock or your device's clock for the most accurate and up-to-date information.", object='entry', type='message.output', created_at=datetime.datetime(2026, 6, 16, 16, 5, 14, 159941, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 6, 16, 16, 5, 14, 400952, tzinfo=TzInfo(0)), agent_id=Unset(), model='mistral-small-latest', id='msg_019ed12e032f7393bbc672481c86771a', role='assistant')]

In [8]:
from mistralai.client.models import CodeInterpreterTool

# Try again with tools
code_response = mistral.beta.conversations.start(
    model="mistral-small-latest",
    inputs="Tell me what is the current time in Paris",
    instructions="Use the code interpreter tool when you have to run code, such as checking current time",
    tools=[CodeInterpreterTool()] # or {"type": "code_interpreter"}
)

code_response.outputs

[ToolExecutionEntry(name='code_interpreter', arguments='{"code": "from datetime import datetime\\n\\n# Get the current time in Paris (Central European Time, UTC+1 or UTC+2 depending on daylight saving time)\\ncurrent_time_paris = datetime.now().astimezone()\\n\\n# Format the time for display\\nformatted_time = current_time_paris.strftime(\'%Y-%m-%d %H:%M:%S %Z\')\\n\\nformatted_time"}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 6, 16, 16, 5, 49, 238961, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 6, 16, 16, 5, 51, 879215, tzinfo=TzInfo(0)), agent_id=Unset(), model='mistral-small-latest', id='tool_exec_019ed12e8c367701a313597a4c75da0a', info={'result': [{'type': 'text', 'text': "'2026-06-16 16:05:50 UTC'\n"}], 'code': "from datetime import datetime\n\n# Get the current time in Paris (Central European Time, UTC+1 or UTC+2 depending on daylight saving time)\ncurrent_time_paris = datetime.now().astimezone()\n\n# Format the time for display\nforma

In [9]:
display_entries(code_response.outputs)

**🤖 ASSISTANT** ----------------------------------------------------------------------------------------------------

The current time in Paris is **18:05:50 CEST** (Central European Summer Time, UTC+2) on June 16, 2026.

```text from datetime import datetime

# Get the current time in Paris (Central European Time, UTC+1 or UTC+2 depending on daylight saving time)
current_time_paris = datetime.now().astimezone()

# Format the time for display
formatted_time = current_time_paris.strftime('%Y-%m-%d %H:%M:%S %Z')

formatted_time

In [10]:
# Example involving file download
chart_response = mistral.beta.conversations.start(
    model="mistral-small-latest",
    inputs="Create a y=2x chart for x between 0 and 10",
    instructions="Use the code interpreter tool when you have to run code",
    tools=[CodeInterpreterTool()]
)

chart_response.outputs

[ToolExecutionEntry(name='code_interpreter', arguments='{"code": "import matplotlib.pyplot as plt\\nimport numpy as np\\n\\n# Define the function\\ndef y(x):\\n    return 2 * x\\n\\n# Generate x values from 0 to 10\\nx_values = np.linspace(0, 10, 100)\\ny_values = y(x_values)\\n\\n# Plot the function\\nplt.figure(figsize=(8, 6))\\nplt.plot(x_values, y_values, label=\'y = 2x\', color=\'blue\')\\nplt.title(\'Graph of y = 2x\')\\nplt.xlabel(\'x\')\\nplt.ylabel(\'y\')\\nplt.grid(True)\\nplt.legend()\\nplt.xlim(0, 10)\\nplt.ylim(0, 20)\\n\\n# Save the plot\\nplt.savefig(\'y_2x_chart.png\', format=\'PNG\', dpi=200, bbox_inches=\'tight\')\\n\\nplt.show()"}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 6, 16, 16, 7, 18, 728828, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 6, 16, 16, 7, 21, 192000, tzinfo=TzInfo(0)), agent_id=Unset(), model='mistral-small-latest', id='tool_exec_019ed12fe9c872d19e4e13c474d87b23', info={'result': [{'type': 'text', 'text': 

In [11]:
display_entries(chart_response.outputs, download_files=True)

**🤖 ASSISTANT** ----------------------------------------------------------------------------------------------------

Here is the chart for the function \( y = 2x \) with \( x \) ranging from 0 to 10. You can download it using the link below:

[Download the y=2x chart](sandbox/y_2x_chart.png)

```text import matplotlib.pyplot as plt
import numpy as np

# Define the function
def y(x):
    return 2 * x

# Generate x values from 0 to 10
x_values = np.linspace(0, 10, 100)
y_values = y(x_values)

# Plot the function
plt.figure(figsize=(8, 6))
plt.plot(x_values, y_values, label='y = 2x', color='blue')
plt.title('Graph of y = 2x')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.legend()
plt.xlim(0, 10)
plt.ylim(0, 20)

# Save the plot
plt.savefig('y_2x_chart.png', format='PNG', dpi=200, bbox_inches='tight')

plt.show()

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  7.4 Image Generation (built-in)
</h3>

In [12]:
from mistralai.client.models import ImageGenerationTool

img_response = mistral.beta.conversations.start(
    model="mistral-small-latest",
    inputs="Create a pixel art image of a fat cat",
    instructions="Use the image generation tool when the user asks for art work",
    tools=[ImageGenerationTool()] # or {"type": "image_generation"}
)

img_response.outputs

[ToolExecutionEntry(name='image_generation', arguments='{"prompt": "A cute fat cat sitting comfortably, pixel art style, 16-bit aesthetic, vibrant colors, clean pixelated edges, soft shading, round and chubby body, big eyes, sitting pose, game-like background with simple patterns"}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 6, 16, 16, 10, 16, 357175, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 6, 16, 16, 10, 23, 569124, tzinfo=TzInfo(0)), agent_id=Unset(), model='mistral-small-latest', id='tool_exec_019ed1329fa5728fab6e0abb86ff8ffc', info={'result': '{"url": "https://mistralaiblackforestprod.blob.core.windows.net/images/blackforest/7e02/687e/-dd6/f-4c25-9490-40ad4b786d38/image.jpg?se=2026-06-16T17%3A10%3A23Z&sp=r&sv=2026-02-06&sr=b&skoid=8aae9820-8683-45ec-b557-e441def5aa94&sktid=4fbc1168-2984-4d17-af19-ac5138c2378e&skt=2026-06-16T16%3A10%3A23Z&ske=2026-06-16T17%3A10%3A23Z&sks=b&skv=2026-02-06&sig=mtxT5wF/d47ijDC/ofTwCYvAZ4pEbKGbsM8xRO%2Blrv

In [13]:
display_entries(img_response.outputs, download_files=True)

**🤖 ASSISTANT** ----------------------------------------------------------------------------------------------------

Here is your pixel art image of a fat cat!



In [14]:
# As usual, you can append to the conversation
# If you append a response to the conversation, there is no need to provide the tools again (they are saved in the ModelConversation object)
new_img_response = mistral.beta.conversations.append(
    conversation_id=img_response.conversation_id,
    inputs="FATTER!"
)

display_entries(new_img_response.outputs, download_files=True)

**🤖 ASSISTANT** ----------------------------------------------------------------------------------------------------

Here’s your **extra chubby** pixel art cat! Hope you like it! 😺🐱‍🏍️

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  7.5 Functions Calling
</h3>

In [15]:
# Create a function
def adder(x: int, y: int) -> int:
    return x+y

# Document it with the standard OpenAI format
toolbox=[
        {
        "type": "function",
        "function": {
            "name":"adder",
            "func": adder, # Added this key to easily create the tools registry below
            "description":"Returns the sum of 2 integers",
            "parameters":{
                "type": "object",
                "properties": {
                    "x": {
                        "type": "number",
                        "description": "The first integer to sum-up",
                    },
                    "y": {
                        "type": "number",
                        "description": "The second integer to sum-up",
                    }
                },
                "required": ["x", "y"]
            }
        }
    }
]

# Create a dynamic tool registry {"function_name1": function1, "function_name2": function2} for simple execution
tools_registry = {tool["function"]["name"]: tool["function"]["func"] for tool in toolbox}

# Start a new conversation
function_response = mistral.beta.conversations.start(
    model="mistral-small-latest",
    instructions="You are a helpful assistant using tools to respond to questions",
    inputs="How much is 7+876?",
    tools=toolbox
)

# The FunctionCallEntry is similar to a call tool in Chat Completions
function_response.outputs

[FunctionCallEntry(tool_call_id='adp65EJTf', name='adder', arguments='{"x": 7, "y": 876}', object='entry', type='function.call', created_at=datetime.datetime(2026, 6, 16, 16, 13, 51, 528429, tzinfo=TzInfo(0)), completed_at=None, agent_id=Unset(), model='mistral-small-latest', id='fc_019ed135e828746b8cbedb7d4a46417c', confirmation_status=None)]

In [16]:
from mistralai.client.models import FunctionResultEntry

# We create a list to store function result entries , which we will append to the conversation
additional_entries = []

for entry in function_response.outputs:

    # We are only interested in tool calls
    if entry.type=="function.call":
        args = json.loads(entry.arguments)
        output = tools_registry[entry.name](**args)

        # Execute function and store result in a FunctionResultEntry (similar to a ToolMessage for Chat Completions)
        additional_entries.append(
            FunctionResultEntry(
                tool_call_id=entry.tool_call_id,
                result=str(output)
            )
        )

# Append to the conversation and get a response
new_function_response = mistral.beta.conversations.append(
        conversation_id=function_response.conversation_id,
        inputs=additional_entries
    )

mistral.beta.conversations.get_history(conversation_id=function_response.conversation_id).entries

[MessageInputEntry(role='user', content='How much is 7+876?', object='entry', type='message.input', created_at=datetime.datetime(2026, 6, 16, 16, 13, 51, 233481, tzinfo=TzInfo(0)), completed_at=None, id='msg_019ed135e70173788edcc9443d9b2ea8', prefix=False),
 FunctionCallEntry(tool_call_id='adp65EJTf', name='adder', arguments='{"x": 7, "y": 876}', object='entry', type='function.call', created_at=datetime.datetime(2026, 6, 16, 16, 13, 51, 528429, tzinfo=TzInfo(0)), completed_at=None, agent_id=Unset(), model='mistral-small-latest', id='fc_019ed135e828746b8cbedb7d4a46417c', confirmation_status=None),
 FunctionResultEntry(tool_call_id='adp65EJTf', result='883', object='entry', type='function.result', created_at=datetime.datetime(2026, 6, 16, 16, 17, 31, 105514, tzinfo=TzInfo(0)), completed_at=None, id='fc_res_019ed13941e171218a5826f0635e321b'),
 MessageOutputEntry(content='The result of 7 + 876 is **883**.', object='entry', type='message.output', created_at=datetime.datetime(2026, 6, 16, 16

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  7.6 Resources Clean-Up
</h3>

In [17]:
# Delete conversations
for conv in mistral.beta.conversations.list():
    mistral.beta.conversations.delete(conversation_id=conv.id)

mistral.beta.conversations.list()

[]

In [18]:
# Delete files
for file in mistral.files.list().data:
    mistral.files.delete(file_id=file.id)

mistral.files.list().data

[]